In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
INTERIM = PROJECT_ROOT / "data" / "interim"

DATA_FILE = INTERIM / "cafe_eda.csv"

cafe_df = pd.read_csv(DATA_FILE)

print("Shape:", cafe_df.shape)
cafe_df.head()

Shape: (552, 24)


,name,url,location,din_rev,del_rev,category,cuisine_offered,cost_for_two,is_vegeterian,is_bar_available,...,is_explicit_cafe,has_cafe_type_cuisine,has_cafe_name_signal,is_cafescope_cafe,dining_review_count,delivery_review_count,price_segment,popularity_segment,rating_segment,market_segment
0,11MG,https://www.zomato.com/jaipur/11mg-mansarovar,Mansarovar,135 Dining Reviews,837 Delivery Reviews,CAFE,Cafe,400.0,True,False,...,True,True,False,True,135.0,837.0,Mid-range,High Popularity,High Rating,High Rating + High Popularity
1,365 Cafe,https://www.zomato.com/jaipur/365-cafe-malviya...,Malviya Nagar,Not enough Dining Reviews,Does not offer Delivery,Quick Bites,"Fast Food, Beverages",400.0,True,False,...,False,True,True,True,NaN,NaN,Mid-range,Low Popularity,Low Rating,Low Rating + Low Popularity
2,4 Friends Cafe,https://www.zomato.com/jaipur/4-friends-cafe-t...,Durgapura,Not enough Dining Reviews,Not enough Delivery Reviews,Quick Bites,Fast Food,150.0,False,False,...,False,False,True,True,NaN,NaN,Budget,Low Popularity,Low Rating,Low Rating + Low Popularity
3,8 Fumes,https://www.zomato.com/jaipur/8-fumes-mansarovar,Mansarovar,8 Dining Reviews,Does not offer Delivery,CAFE,"Cafe, Fast Food",400.0,True,False,...,True,True,False,True,8.0,NaN,Mid-range,Low Popularity,Low Rating,Low Rating + Low Popularity
4,999.pt Cafe - Restaurant,https://www.zomato.com/jaipur/999-pt-cafe-rest...,Pratap Nagar,111 Dining Reviews,"2,537 Delivery Reviews",Quick Bites,"North Indian, Fast Food",300.0,False,False,...,False,False,True,True,111.0,2537.0,Budget,High Popularity,Low Rating,Low Rating + High Popularity


In [16]:
cafe_df.columns.tolist()

['name',
 'url',
 'location',
 'din_rev',
 'del_rev',
 'category',
 'cuisine_offered',
 'cost_for_two',
 'is_vegeterian',
 'is_bar_available',
 'home_delivery',
 'offers_dining',
 'avg_din_rate',
 'avg_del_rate',
 'is_explicit_cafe',
 'has_cafe_type_cuisine',
 'has_cafe_name_signal',
 'is_cafescope_cafe',
 'dining_review_count',
 'delivery_review_count',
 'price_segment',
 'popularity_segment',
 'rating_segment',
 'market_segment']

In [17]:
bins = [0, 300, 500, 700, 1000]
labels = ["Budget", "Mid-range", "Upper-mid", "Premium"]

cafe_df["price_segment"] = pd.cut(
    cafe_df["cost_for_two"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

cafe_df["price_segment"].value_counts().sort_index()

price_segment
Budget       197
Mid-range    207
Upper-mid     88
Premium       35
Name: count, dtype: int64

In [18]:
cafe_df["cuisine_count"] = (
    cafe_df["cuisine_offered"]
    .fillna("")
    .apply(
        lambda x: len([
            cuisine.strip()
            for cuisine in x.split(",")
            if cuisine.strip() and cuisine.strip() != "Unknown"
        ])
    )
)

cafe_df[["name", "cuisine_offered", "cuisine_count"]].head(10)

,name,cuisine_offered,cuisine_count
0,11MG,Cafe,1
1,365 Cafe,"Fast Food, Beverages",2
2,4 Friends Cafe,Fast Food,1
3,8 Fumes,"Cafe, Fast Food",2
4,999.pt Cafe - Restaurant,"North Indian, Fast Food",2
5,A.D. Cafe,Fast Food,1
6,A2G All Together,"Cafe, Desserts, North Indian, Chinese, Frozen ...",5
7,Aapno Cafe,"Beverages, Fast Food",2
8,Affetto Cafe,"North Indian, Chinese, Fast Food, Beverages",4
9,After 8,"Finger Food, Continental, Beverages",3


In [19]:
cafe_df["log_dining_reviews"] = np.log1p(
    cafe_df["dining_review_count"]
)

cafe_df[
    ["dining_review_count", "log_dining_reviews"]
].head(10)

,dining_review_count,log_dining_reviews
0,135.0,4.912655
1,NaN,NaN
2,NaN,NaN
3,8.0,2.197225
4,111.0,4.718499
5,NaN,NaN
6,496.0,6.208590
7,12.0,2.564949
8,154.0,5.043425
9,NaN,NaN


In [20]:
cafe_df["has_delivery_review_data"] = (
    cafe_df["delivery_review_count"].notna()
)

In [21]:
cafe_df["cuisine_count"].describe()

count    552.000000
mean       2.764493
std        1.361503
min        0.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        8.000000
Name: cuisine_count, dtype: float64

In [22]:
cafe_df["cuisine_count"].value_counts().sort_index()

cuisine_count
0      1
1     90
2    180
3    142
4     77
5     37
6     20
7      3
8      2
Name: count, dtype: int64

## Feature Engineering Decisions

| Feature | Type | Purpose | Why created |
|---|---|---|---|
| price_segment | Categorical | Price comparison | Converts continuous price into interpretable market segments |
| cuisine_count | Numeric | Menu breadth | Counts cuisines offered by each cafe |
| log_dining_reviews | Numeric | Review-volume analysis | Reduces skew in highly uneven review counts |
| has_delivery_review_data | Boolean | Data availability | Distinguishes available delivery-review data from missing data |

### Deliberately not created

- Visibility score — review count is only a proxy; no need for an arbitrary score yet.
- Underrated score — requires a justified methodology.
- Operational score — no clear analytical question currently requires it.
- Area-level cafe count — better calculated through aggregation in SQL/Power BI.

In [23]:
feature_columns = [
    "price_segment",
    "cuisine_count",
    "log_dining_reviews",
    "has_delivery_review_data"
]

cafe_df[feature_columns].head(10)

,price_segment,cuisine_count,log_dining_reviews,has_delivery_review_data
0,Mid-range,1,4.912655,True
1,Mid-range,2,NaN,False
2,Budget,1,NaN,False
3,Mid-range,2,2.197225,False
4,Budget,2,4.718499,True
5,Budget,1,NaN,False
6,Upper-mid,5,6.208590,True
7,Budget,2,2.564949,False
8,Upper-mid,4,5.043425,True
9,NaN,3,NaN,False


In [24]:
feature_output = PROCESSED_DATA_DIR / "cafe_feature_eng.csv"
cafe_df.to_csv(feature_output, index=False)
print("Feature dataset saved:", feature_output)

Feature dataset saved: /Users/priyashekhawat/Desktop/ree/project/cafescope/data/processed/cafe_feature_eng.csv
